原有老师参考代码完整保留；末尾补充样卷要求的可视化。

In [ ]:
import pandas as pd

# 读取数据
bookings = pd.read_csv('meeting_bookings.csv', encoding='utf-8-sig')
rooms = pd.read_csv('meeting_rooms.csv', encoding='utf-8-sig')

# M1（1分）：数据准备
# 按booking_id去重，再排除已取消预约
clean = bookings.drop_duplicates(subset='booking_id')
valid = clean[clean['status'] != '已取消']

print('数据准备结果')
print('去重后预约数：', len(clean))
print('有效预约数：', len(valid))

# M2（2分）：识别爽约高发时段
# 统计各时段的有效预约数、爽约数和爽约率，并按爽约率降序排列
valid['is_no_show'] = (valid['status'] == '爽约').astype(int)
slot_summary = valid.groupby('time_slot').agg(有效预约数=('is_no_show','count'), 爽约数=('is_no_show','sum'), 爽约率=('is_no_show','mean')).sort_values('爽约率', ascending=False)

print('\n各时段爽约情况')
print(slot_summary)

# M3（2分）：分析会议室容量匹配
# 只分析已使用预约，关联会议室容量，统计使用场次和平均座位利用率
used = valid[valid['status'] == '已使用'].merge(rooms, on='room_id')
used['seat_util'] = used["attendees"] / used["capacity"]
room_summary = used.groupby(['room_id','room_name','capacity']).agg(使用场次=('booking_id','count'), 平均座位利用率=('seat_util','mean')).reset_index()

print('\n各会议室空间利用情况')
print(room_summary)

# M4（1分）：输出空间匹配优化候选清单
# 筛选容量不少于12、实际使用不少于20场、平均座位利用率低于40%的会议室
candidates = room_summary[(room_summary['capacity'] >= 12) & (room_summary['使用场次'] >= 20) & (room_summary['平均座位利用率'] < 0.4)]

print('\n空间匹配优化候选清单')
print(candidates)

In [ ]:
# 样卷任务一补充：数据可视化（老师参考代码保持原样）
import matplotlib.pyplot as plt

# 对比各时段爽约率，标出需要优先干预的时段
slot_labels = {'上午': 'Morning', '午间': 'Midday', '下午': 'Afternoon', '傍晚': 'Evening'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([slot_labels.get(str(x), str(x)) for x in slot_summary.index], slot_summary['爽约率'] * 100)
axes[0].set(title='No-show rate by time slot', ylabel='No-show rate (%)')

# 对比已使用场次的平均座位利用率；低于40%为优化候选
axes[1].bar(room_summary['room_id'].astype(str), room_summary['平均座位利用率'] * 100)
axes[1].axhline(40, color='red', linestyle='--', label='40% threshold')
axes[1].set(title='Mean seat utilization by room', ylabel='Seat utilization (%)', xlabel='Room')
axes[1].legend()
plt.tight_layout()
